## Overview of selectExpr on Spark Data Frame

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Overview of selectExpr on Spark Data Frame").getOrCreate()

24/02/22 19:30:43 WARN Utils: Your hostname, Qasim resolves to a loopback address: 127.0.1.1; using 172.30.54.131 instead (on interface eth0)
24/02/22 19:30:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/02/22 19:30:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/02/22 19:30:59 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/02/22 19:30:59 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
24/02/22 19:30:59 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


In [2]:
import datetime
users_list = [
    {
        'id': 1,
        'first_name': 'John',
        'last_name': 'Doe',
        'email': 'john.doe@example.com',
        'phone_numer' : {'UAE_Mobile': '+971 54 7400113', 'PAK_Mobile': '+92 300 5638080'},
        'is_customer': True,
        'amount_paid': 123.45,
        'customer_from': datetime.date(2023, 1, 15),  # Date in YYYY-MM-DD format
        'last_updated_ts': datetime.datetime(2024, 1, 21, 20, 5, 46)  # Timestamp in YYYY-MM-DD HH:MM:SS format
    },
    {
        'id': 2,
        'first_name': 'Jane',
        'last_name': 'Smith',
        'email': 'jane.smith@example.com',
        'phone_numer' : {'UAE_Mobile': '+971 99 7666666', 'PAK_Mobile': '+92 300 0000000'},
        'is_customer': True,
        'amount_paid': 0.0,
        'customer_from': datetime.date(2022, 11, 10),
        'last_updated_ts': datetime.datetime(2024, 1, 21, 20, 5, 46)
    },
    {
        'id': 3,
        'first_name': 'Jack',
        'last_name': 'Alpha',
        'email': 'jack.alpha@example.com',
        'phone_numer' : {'UAE_Mobile': '+971 99 7666555'},
        'is_customer': False,
        'amount_paid': 0.0,
        'customer_from': datetime.date(2022, 11, 10),
        'last_updated_ts': datetime.datetime(2024, 1, 21, 20, 5, 46)
    }
    # Add more user dictionaries as needed
]

In [3]:
from pyspark.sql import Row

In [4]:
users_df = spark.createDataFrame(Row(**users) for users in users_list)
users_df

DataFrame[id: bigint, first_name: string, last_name: string, email: string, phone_numer: map<string,string>, is_customer: boolean, amount_paid: double, customer_from: date, last_updated_ts: timestamp]

In [5]:
help(users_df.selectExpr)

Help on method selectExpr in module pyspark.sql.dataframe:

selectExpr(*expr: Union[str, List[str]]) -> 'DataFrame' method of pyspark.sql.dataframe.DataFrame instance
    Projects a set of SQL expressions and returns a new :class:`DataFrame`.
    
    This is a variant of :func:`select` that accepts SQL expressions.
    
    .. versionadded:: 1.3.0
    
    .. versionchanged:: 3.4.0
        Supports Spark Connect.
    
    Returns
    -------
    :class:`DataFrame`
        A DataFrame with new/old columns transformed by expressions.
    
    Examples
    --------
    >>> df = spark.createDataFrame([
    ...     (2, "Alice"), (5, "Bob")], schema=["age", "name"])
    >>> df.selectExpr("age * 2", "abs(age)").show()
    +---------+--------+
    |(age * 2)|abs(age)|
    +---------+--------+
    |        4|       2|
    |       10|       5|
    +---------+--------+



In [6]:
users_df.selectExpr('*').show()

+---+----------+---------+--------------------+--------------------+-----------+-----------+-------------+-------------------+
| id|first_name|last_name|               email|         phone_numer|is_customer|amount_paid|customer_from|    last_updated_ts|
+---+----------+---------+--------------------+--------------------+-----------+-----------+-------------+-------------------+
|  1|      John|      Doe|john.doe@example.com|{PAK_Mobile -> +9...|       true|     123.45|   2023-01-15|2024-01-21 20:05:46|
|  2|      Jane|    Smith|jane.smith@exampl...|{PAK_Mobile -> +9...|       true|        0.0|   2022-11-10|2024-01-21 20:05:46|
|  3|      Jack|    Alpha|jack.alpha@exampl...|{UAE_Mobile -> +9...|      false|        0.0|   2022-11-10|2024-01-21 20:05:46|
+---+----------+---------+--------------------+--------------------+-----------+-----------+-------------+-------------------+



In [8]:
users_df.alias('ud').selectExpr('ud.id', 'ud.first_name', 'ud.last_name').show()

+---+----------+---------+
| id|first_name|last_name|
+---+----------+---------+
|  1|      John|      Doe|
|  2|      Jane|    Smith|
|  3|      Jack|    Alpha|
+---+----------+---------+



In [9]:
users_df.alias('ud').selectExpr(['ud.id', 'ud.first_name', 'ud.last_name']).show()

+---+----------+---------+
| id|first_name|last_name|
+---+----------+---------+
|  1|      John|      Doe|
|  2|      Jane|    Smith|
|  3|      Jack|    Alpha|
+---+----------+---------+



In [17]:
# using single quotes while concating as string is not allowed
# selectExpr to combine Spark.SQL functions along with datafram APIs

users_df.selectExpr('id', 'first_name', 'last_name', "concat(first_name, ', ', last_name) as full_name").show()

+---+----------+---------+-----------+
| id|first_name|last_name|  full_name|
+---+----------+---------+-----------+
|  1|      John|      Doe|  John, Doe|
|  2|      Jane|    Smith|Jane, Smith|
|  3|      Jack|    Alpha|Jack, Alpha|
+---+----------+---------+-----------+



In [15]:
users_df.createOrReplaceTempView('users_vw')

In [16]:
spark.sql("""
    SELECT id, first_name, last_name, concat(first_name, ', ', last_name) from users_vw  
"""
    ). \
    show()

+---+----------+---------+---------------------------------+
| id|first_name|last_name|concat(first_name, , , last_name)|
+---+----------+---------+---------------------------------+
|  1|      John|      Doe|                        John, Doe|
|  2|      Jane|    Smith|                      Jane, Smith|
|  3|      Jack|    Alpha|                      Jack, Alpha|
+---+----------+---------+---------------------------------+

